# AER - Access Entitlement Review Reporter v4.2

**Updates (v4.2):**
- **Bug Fix**: Cache hit stats no longer inflate (per-row calculation instead of total).
- **Global Report**: New `📊 Global Report` button exports summary across all apps.
- **User Info**: Per-App reports now include User Name & User Email columns.
- **File Paths**: All reports saved under `output/{date}/report/`, logs under `report/logs/`.
- **Safe Write**: Auto-appends `_1`, `_2`... if target file is locked/open.

**Previous Features:**
- **Smart Tab Handling**: Prioritizes "User Listing" tab.
- **Full Data Caching**: Reports populated from cache are identical to live reads.
- **Merged Logic**: Parsing logic integrated into Engine.
- **Full Audit History & Advanced Reporting**.

In [ ]:
# === Cell 1: Setup & Authentication ===
import os
import sys
import io
import logging
from datetime import datetime
from dotenv import load_dotenv
from msal import PublicClientApplication
import requests

load_dotenv()

# === Logging ===
today_str = datetime.now().strftime('%Y-%m-%d')
hour_str = datetime.now().strftime('%H')
log_dir = f"output/{today_str}/report/logs"
os.makedirs(log_dir, exist_ok=True)

log_file = f"{log_dir}/aer_{today_str}_{hour_str}00.log"

logger = logging.getLogger("aer")
logger.handlers.clear()
logger.setLevel(logging.INFO)

def _get_console_stream():
    s = getattr(sys, "stdout", None)
    try:
        if s and hasattr(s, "reconfigure"):
            s.reconfigure(encoding="utf-8")
            if hasattr(sys.stderr, "reconfigure"):
                sys.stderr.reconfigure(encoding="utf-8")
            return s
    except Exception: pass
    return sys.stdout

formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
ch = logging.StreamHandler(_get_console_stream())
ch.setFormatter(formatter)
logger.addHandler(ch)

fh = logging.FileHandler(log_file, encoding="utf-8", mode='a')
fh.setFormatter(formatter)
logger.addHandler(fh)

# === Azure AD Config ===
TENANT_ID = os.getenv("AZURE_TENANT_ID")
CLIENT_ID = os.getenv("AZURE_CLIENT_ID")
SHAREPOINT_HOST = os.getenv("SHAREPOINT_HOST", "davidshih.sharepoint.com")
SITE_NAME = os.getenv("SITE_NAME", "aer")
APP_NAME = "2025 Entitlement Review"
BASE_PATH = APP_NAME
SENDER_EMAIL = os.getenv("SENDER_EMAIL")

SCOPES = [
    "User.Read.All", "Mail.Send", "Mail.Read", 
    "Files.Read.All", "Sites.Read.All"
]

app = PublicClientApplication(CLIENT_ID, authority=f"https://login.microsoftonline.com/{TENANT_ID}")
print("🚀 Opening browser for authentication...")
interactive_result = app.acquire_token_interactive(scopes=SCOPES, prompt="select_account")

if "access_token" not in interactive_result:
    raise RuntimeError(f"Login Failed: {interactive_result.get('error_description')}")

headers = {"Authorization": f"Bearer {interactive_result['access_token']}"}
logger.info("✅ Login Successful")
logger.info(f"Log File: {log_file}")

In [ ]:
# === Cell 2: SharePoint API & App Selector (v3.6 with Counts & Filtering) ===
import ipywidgets as widgets
from IPython.display import display, clear_output
import requests

# --- 1. SharePoint API Functions ---
def get_site_id(site_name: str) -> str:
    url = f"https://graph.microsoft.com/v1.0/sites/{SHAREPOINT_HOST}:/sites/{site_name}"
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    return resp.json()["id"]

def list_folders_with_count(site_id: str, path: str) -> list[dict]:
    if not path or path.strip() == "":
        url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/root/children"
    else:
        clean_path = path.strip("/")
        url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/root:/{clean_path}:/children"
    
    resp = requests.get(url, headers=headers)
    results = []
    EXCLUDED_NAMES = ["forms", "_private", "audit logs", "audit", "user listings", "shared documents"]

    for item in resp.json().get("value", []):
        if item.get("folder"):
            name_lower = item["name"].lower()
            if any(ex in name_lower for ex in EXCLUDED_NAMES):
                continue

            count = item.get("folder", {}).get("childCount", 0)
            results.append({
                "name": item["name"], 
                "webUrl": item.get("webUrl", ""),
                "count": count
            })
    return results

def list_excel_files(site_id: str, folder_path: str) -> list[dict]:
    clean_path = folder_path.strip("/")
    url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/root:/{clean_path}:/children"
    resp = requests.get(url, headers=headers)
    files = []
    for item in resp.json().get("value", []):
        if item["name"].endswith(".xlsx"):
            files.append({
                "id": item["id"], "name": item["name"],
                "lastModifiedDateTime": item.get("lastModifiedDateTime"),
                "createdDateTime": item.get("createdDateTime"),
                "webUrl": item.get("webUrl")
            })
    return sorted(files, key=lambda f: f.get("lastModifiedDateTime", ""), reverse=True)

def download_file(site_id: str, file_path: str) -> bytes:
    clean_path = file_path.strip("/")
    url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/root:/{clean_path}:/content"
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    return resp.content

def get_file_audit_info(site_id: str, file_id: str) -> dict:
    url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/items/{file_id}/versions"
    resp = requests.get(url, headers=headers)
    default_res = {"log": "N/A", "creator": "Unknown", "modifier": "Unknown", "created_ts": None}
    if resp.status_code != 200: return default_res
    
    versions = resp.json().get("value", [])
    if not versions: return default_res

    logs = []
    for v in versions:
        mod_time = v.get("lastModifiedDateTime", "")[:19].replace("T", " ")
        actor = v.get("lastModifiedBy", {}).get("user", {}).get("displayName") or "System"
        logs.append(f"{mod_time} - {actor}")
    
    last_v = versions[0]
    first_v = versions[-1]

    return {
        "log": "\n".join(logs),
        "creator": first_v.get("lastModifiedBy", {}).get("user", {}).get("displayName") or "System",
        "modifier": last_v.get("lastModifiedBy", {}).get("user", {}).get("displayName") or "System",
        "created_ts": first_v.get("lastModifiedDateTime")
    }

# --- 2. Initialize Connection ---
try:
    site_id = get_site_id(SITE_NAME)
    logger.info(f"✅ SharePoint Connected (Site ID: {site_id[:10]}...)")
except Exception as e:
    logger.error(f"❌ Connection Failed: {e}")

# --- 3. Tree Selector UI (with Counts) ---
TARGET_APPS = [] 
USE_CACHE = True 

def create_app_selector():
    print(f"📂 Reading Root: {BASE_PATH} ...")
    try:
        categories = list_folders_with_count(site_id, BASE_PATH)
    except Exception as e:
        print(f"❌ Read Failed: {e}")
        return

    ui_container = widgets.VBox()
    app_checkboxes = []
    app_checkbox_map = {}
    
    chk_cache = widgets.Checkbox(value=True, description="⚡ Use Cache (Faster)", indent=False)
    ui_container.children += (widgets.HTML("<h3>📂 App Selector (v3.6)</h3>"), chk_cache, widgets.HTML("<hr>"))

    for cat in categories:
        if cat['name'] in ["Forms", "_private"] or "audit" in cat['name'].lower(): continue
            
        cat_label = widgets.HTML(f"<b>📁 {cat['name']}</b>", layout=widgets.Layout(width='150px'))
        btn_expand = widgets.Button(description="➕ Expand", button_style='info', layout=widgets.Layout(width='80px'))
        pbar = widgets.IntProgress(value=0, min=0, max=1, layout=widgets.Layout(width='160px', visibility='hidden'))
        app_list_box = widgets.VBox(layout=widgets.Layout(margin='0 0 0 30px', display='none'))
        
        def on_expand(b, cat_name=cat['name'], container=app_list_box, btn=btn_expand, pbar=pbar):
            if btn.description == "➕ Expand":
                btn.description = "⏳"
                pbar.layout.visibility = 'visible'
                pbar.bar_style = 'info'
                pbar.value = 0
                pbar.max = 1
                sub_path = f"{BASE_PATH}/{cat_name}"
                try:
                    apps = list_folders_with_count(site_id, sub_path)
                    apps = [a for a in apps if a.get('name') not in ["Forms", "_private"]]
                    checks = []
                    if not apps:
                        checks.append(widgets.Label("(Empty)"))
                        pbar.value = 1
                    else:
                        pbar.max = len(apps)
                        for idx, app in enumerate(apps, 1):
                            count_display = f" <span style='color:#888; font-size:11px'>({app['count']} users)</span>"
                            app_key = f"{cat_name}|{app['name']}"
                            cb = app_checkbox_map.get(app_key)
                            if cb is None:
                                cb = widgets.Checkbox(value=False, description=app['name'], indent=False, layout=widgets.Layout(width='400px'))
                                app_checkbox_map[app_key] = cb
                                app_checkboxes.append(cb)
                            cb.description = app['name']
                            cb.app_data = (cat_name, app['name'], f"{sub_path}/{app['name']}")
                            lbl_count = widgets.HTML(count_display)
                            checks.append(widgets.HBox([cb, lbl_count]))
                            pbar.value = idx
                    container.children = tuple(checks)
                    container.layout.display = 'block'
                    btn.description = "➖ Collapse"
                except Exception as e:
                    container.children = (widgets.Label(f"Error: {e}"),)
                    btn.description = "❌"
                finally:
                    pbar.layout.visibility = 'hidden'
                    pbar.bar_style = ''
            else:
                if container.layout.display == 'none':
                    container.layout.display = 'block'; btn.description = "➖ Collapse"
                else:
                    container.layout.display = 'none'; btn.description = "➕ Expand"

        btn_expand.on_click(on_expand)
        ui_container.children += (widgets.HBox([btn_expand, cat_label, pbar]), app_list_box)

    btn_confirm = widgets.Button(description="✅ Confirm Selection", button_style='success', layout=widgets.Layout(margin='20px 0'))
    output_area = widgets.Output()

    def on_confirm(b):
        global TARGET_APPS, USE_CACHE
        TARGET_APPS = [cb.app_data for cb in app_checkboxes if cb.value]
        USE_CACHE = chk_cache.value
        
        with output_area:
            clear_output()
            if not TARGET_APPS: print("⚠️ No apps selected!")
            else:
                print(f"🎯 Selected {len(TARGET_APPS)} Apps | Cache: {USE_CACHE}")
                print("⏳ Proceed to Cell 4 (Enrichment) or Cell 5 (Scan).")

    btn_confirm.on_click(on_confirm)
    display(ui_container, btn_confirm, output_area)

create_app_selector()

In [ ]:
# === Cell 5: AER Engine v4.2 (Cache Fix, User Info, Global Report, Safe Write) ===

import pandas as pd

import re, time, json, os, requests

import ipywidgets as widgets

from datetime import datetime

from openpyxl import load_workbook
from openpyxl.styles import Alignment
from openpyxl.utils import get_column_letter

from io import BytesIO

from IPython.display import display, HTML, clear_output



# ==========================================

# PART 1: CONFIG & LOADER

# ==========================================

CACHE_FILE = "aer_cache.json"

NOTES_FILE = "aer_manual_notes.json"

CACHE_VERSION = 4.2

cache_updated = False

EXCLUDED_FOLDERS = ["forms", "_private", "user listings", "audit logs", "audit"]



def load_json(file_path):

    if os.path.exists(file_path):

        try:

            with open(file_path, 'r', encoding='utf-8') as f: return json.load(f)

        except: return {}

    return {}



def save_json(file_path, data):

    with open(file_path, 'w', encoding='utf-8') as f: json.dump(data, f, indent=2, ensure_ascii=False)



def safe_excel_path(base_path):

    """If base_path is locked/open, append _1, _2, ... until writable."""

    if not os.path.exists(base_path):

        return base_path

    try:

        with open(base_path, 'a'):

            pass

        return base_path

    except OSError:

        pass

    stem, ext = os.path.splitext(base_path)

    for i in range(1, 100):

        candidate = f"{stem}_{i}{ext}"

        if not os.path.exists(candidate):

            return candidate

    return f"{stem}_{int(time.time())}{ext}"



def format_export_excel(file_path, audit_col_name="Audit Log"):

    wb = load_workbook(file_path)

    ws = wb.active

    header_to_col = {}

    for col_idx in range(1, ws.max_column + 1):

        hdr = ws.cell(row=1, column=col_idx).value

        hdr_txt = str(hdr).strip() if hdr is not None else ""

        if hdr_txt:

            header_to_col[hdr_txt] = col_idx

        max_len = len(hdr_txt)

        for row_idx in range(2, ws.max_row + 1):

            cell_val = ws.cell(row=row_idx, column=col_idx).value

            if cell_val is None:

                continue

            lines = str(cell_val).splitlines() or [str(cell_val)]

            max_len = max(max_len, max(len(line) for line in lines))

        ws.column_dimensions[get_column_letter(col_idx)].width = min(max(10, max_len + 2), 80)

    audit_col = header_to_col.get(audit_col_name)

    if audit_col:

        for row_idx in range(2, ws.max_row + 1):

            cell = ws.cell(row=row_idx, column=audit_col)

            txt = "" if cell.value is None else str(cell.value)

            line_count = max(1, txt.count("\n") + 1)

            cell.alignment = Alignment(wrap_text=True, vertical="top")

            current_height = ws.row_dimensions[row_idx].height or 15

            ws.row_dimensions[row_idx].height = max(current_height, min(15 * line_count, 300))

    wb.save(file_path)



local_cache = load_json(CACHE_FILE)

manual_data_store = load_json(NOTES_FILE)



# ==========================================

# PART 2: ROBUST EXCEL PARSER

# ==========================================

def find_col_index(headers, keywords):

    """Case-insensitive search for column index."""

    for idx, h in enumerate(headers):

        if not h: continue

        h_str = str(h).strip().lower()

        if all(k in h_str for k in keywords):

            return idx

    return None



def is_name_like_header(header_text: str) -> bool:

    txt = (header_text or "").strip().lower()

    if not txt:

        return False

    if "reviewer" in txt or "manager" in txt:

        return False

    return ("name" in txt) or ("display" in txt and "user" in txt) or ("full" in txt and "name" in txt)



def is_email_like_header(header_text: str) -> bool:

    txt = (header_text or "").strip().lower()

    if not txt:

        return False

    return ("email" in txt) or ("mail" == txt) or txt.endswith(" mail")



def resolve_column_map(header, app_col_map=None):

    if app_col_map:

        return app_col_map, "app-locked"

    idx_rev = find_col_index(header, ["reviewer"])

    idx_res = find_col_index(header, ["response"])

    idx_det = find_col_index(header, ["details", "change"])

    idx_user = None

    idx_email = None

    if idx_rev is not None:

        rev_hdr = str(header[idx_rev]).strip().lower() if header[idx_rev] else ""

        if "response" in rev_hdr:

            idx_rev = None

            for i, h in enumerate(header):

                h_str = str(h).strip().lower() if h else ""

                if ("reviewer" in h_str) and ("response" not in h_str):

                    idx_rev = i

                    break

    for i in range(min(2, len(header))):

        h = str(header[i]).strip().lower() if header[i] else ""

        if idx_user is None and is_name_like_header(h):

            idx_user = i

        if idx_email is None and is_email_like_header(h):

            idx_email = i

    if idx_user is None: idx_user = find_col_index(header, ["user", "name"])

    if idx_user is None: idx_user = find_col_index(header, ["display", "name"])

    if idx_user is None: idx_user = find_col_index(header, ["full", "name"])

    if idx_user is None: idx_user = find_col_index(header, ["name"])

    if idx_email is None: idx_email = find_col_index(header, ["email"])

    if idx_email is None: idx_email = find_col_index(header, ["mail"])

    if idx_rev is None: idx_rev = find_col_index(header, ["manager"])

    if idx_rev is None or idx_res is None:

        return None, "invalid"

    return {

        "idx_rev": idx_rev, "idx_res": idx_res, "idx_det": idx_det,

        "idx_user": idx_user, "idx_email": idx_email

    }, "detected"



def read_excel_rows(excel_bytes: bytes, reviewer_name: str, file_name: str, folder_url: str, app_col_map=None):

    wb = load_workbook(BytesIO(excel_bytes), read_only=True)



    # 1. Smart Tab Selection

    sheet_name = wb.sheetnames[0]

    for sn in wb.sheetnames:

        if "user listing" in sn.lower(): sheet_name = sn; break

    ws = wb[sheet_name]



    # 2. Read Headers

    rows_iter = ws.iter_rows(values_only=True)

    try:

        header = next(rows_iter)

    except StopIteration:

        return [], None, "empty-sheet"



    # 3. Column Mapping

    col_map, map_source = resolve_column_map(header, app_col_map=app_col_map)

    if not col_map:

        return [], None, map_source

    idx_rev = col_map["idx_rev"]

    idx_res = col_map["idx_res"]

    idx_det = col_map["idx_det"]

    idx_user = col_map["idx_user"]

    idx_email = col_map["idx_email"]



    results = []



    # 4. Iterate Rows

    for i, row in enumerate(rows_iter, start=2):

        r_rev = row[idx_rev] if idx_rev < len(row) else None

        r_res = row[idx_res] if idx_res < len(row) else None

        r_det = row[idx_det] if idx_det is not None and idx_det < len(row) else None

        r_user = row[idx_user] if idx_user is not None and idx_user < len(row) else None

        r_email = row[idx_email] if idx_email is not None and idx_email < len(row) else None



        if str(r_rev).strip().lower() != reviewer_name.lower(): continue



        val_res = str(r_res).strip() if r_res else ""

        val_det = str(r_det).strip() if r_det else ""



        results.append({

            "reviewer": reviewer_name,

            "user_name": str(r_user).strip() if r_user else "",

            "user_email": str(r_email).strip() if r_email else "",

            "response": val_res,

            "details": val_det,

            "is_missing": (val_res == ""),

            "row_number": i,

            "file_name": file_name,

            "folder_url": folder_url

        })

    return results, col_map, map_source



def get_row_stats(txt):

    txt = str(txt).lower().strip()

    kw_appr = ['approv', 'retain', 'keep', 'confirm', 'yes', 'ok', 'active']

    kw_deny = ['denied', 'deny', 'remove', 'delete', 'revok', 'reject', 'no']

    kw_chg  = ['change', 'modif', 'updat', 'correct', 'edit', 'adjust']

    return {

        "is_appr": 1 if any(k in txt for k in kw_appr) else 0,

        "is_deny": 1 if any(k in txt for k in kw_deny) else 0,

        "is_chg":  1 if any(k in txt for k in kw_chg) else 0

    }



# ==========================================

# PART 3: SCANNING ENGINE

# ==========================================

if 'TARGET_APPS' not in globals() or not TARGET_APPS:

    print("⚠️ Please select Apps in Cell 2 first!")

    TARGET_APPS = []



all_responses = []

errors = []

app_column_map_store = {}



print(f"🚀 Starting Scan Engine v{CACHE_VERSION} (Fuzzy Column Match)...")



# UI: Per-app scan progress (details stay in log file)
scan_progress_box = widgets.VBox()
scan_progress_store = {}

if TARGET_APPS:
    scan_rows = []
    for category, app_name, app_path in TARGET_APPS:
        ui_key = f"{category}|{app_name}|{app_path}"
        w_label = widgets.HTML(
            f"<b>{category} &gt; {app_name}</b>",
            layout=widgets.Layout(width="360px"),
        )
        w_bar = widgets.IntProgress(
            value=0,
            min=0,
            max=1,
            bar_style="",
            layout=widgets.Layout(width="260px"),
        )
        w_status = widgets.HTML("Queued", layout=widgets.Layout(width="120px"))
        scan_progress_store[ui_key] = {"bar": w_bar, "status": w_status}
        scan_rows.append(
            widgets.HBox([w_label, w_bar, w_status], layout=widgets.Layout(align_items="center"))
        )
    scan_progress_box.children = tuple(scan_rows)
    display(widgets.VBox([widgets.HTML("<h4>📡 Scan Progress (Per App)</h4>"), scan_progress_box]))


for category, current_app_name, current_path in TARGET_APPS:

    ui_key = f"{category}|{current_app_name}|{current_path}"
    app_ui = scan_progress_store.get(ui_key)

    try:

        if app_ui:
            app_ui["bar"].bar_style = "info"
            app_ui["bar"].value = 0
            app_ui["bar"].max = 1
            app_ui["status"].value = "Loading..."

        raw_folders = list_folders_with_count(site_id, current_path) if 'list_folders_with_count' in globals() else []

        if not raw_folders: raw_folders = [{"name": "Error", "webUrl": "#"}]



        reviewers = [r for r in raw_folders if r['name'].lower() not in EXCLUDED_FOLDERS]

        total_revs = len(reviewers)

        if app_ui:
            app_ui["bar"].max = max(1, total_revs)
            app_ui["bar"].value = 0
            app_ui["status"].value = f"0/{total_revs}"

        logger.info(f"📂 App: {current_app_name} | Reviewers: {total_revs}")

        app_schema_key = f"{category}|{current_app_name}"

        app_col_map = app_column_map_store.get(app_schema_key)



        for idx, folder in enumerate(reviewers, 1):

            reviewer_name = folder["name"]

            folder_url = folder["webUrl"]

            folder_path = f"{current_path}/{reviewer_name}"

            cache_key = f"{category}|{current_app_name}|{reviewer_name}"



            try:

                files = list_excel_files(site_id, folder_path)

                match_candidates = [f for f in files if reviewer_name.lower() in f["name"].lower()]

                target_file = match_candidates[0] if match_candidates else (files[0] if files else None)



                if not target_file:

                    logger.info(f"  ⚠️ Skip: [{idx}/{total_revs}] {reviewer_name} (no xlsx found)")

                    continue



                remote_mod = target_file.get("lastModifiedDateTime")

                logger.info(f"  🔎 File: [{idx}/{total_revs}] {reviewer_name} | Target:{target_file['name']} | Modified:{remote_mod}")



                cached = local_cache.get(cache_key)

                is_hit = False
                force_live_recheck = False
                cache_pending = False



                if USE_CACHE and cached and cached.get('v') == CACHE_VERSION and cached.get('last_mod') == remote_mod and 'rows' in cached and len(cached['rows']) > 0:

                    cache_pending = any(r.get('is_missing') for r in cached.get('rows', []))

                    if cache_pending:

                        force_live_recheck = True

                        logger.info(f"  ♻️ Cache Recheck: [{idx}/{total_revs}] {reviewer_name} has pending rows; reading live and overwriting cache.")

                    else:

                        is_hit = True

                        logger.info(f"  🧊 Cache Hit (Completed): [{idx}/{total_revs}] {reviewer_name} | rows:{len(cached['rows'])}")

                else:

                    if not USE_CACHE:

                        logger.info(f"  ℹ️ Cache Bypass: [{idx}/{total_revs}] {reviewer_name} (USE_CACHE=False)")

                    elif not cached:

                        logger.info(f"  ℹ️ Cache Miss: [{idx}/{total_revs}] {reviewer_name} (no cache key)")

                    elif cached.get('v') != CACHE_VERSION:

                        logger.info(f"  ℹ️ Cache Miss: [{idx}/{total_revs}] {reviewer_name} (version {cached.get('v')} != {CACHE_VERSION})")

                    elif cached.get('last_mod') != remote_mod:

                        logger.info(f"  ℹ️ Cache Miss: [{idx}/{total_revs}] {reviewer_name} (modified changed)")

                    else:

                        logger.info(f"  ℹ️ Cache Miss: [{idx}/{total_revs}] {reviewer_name} (no cached rows)")



                if is_hit:

                    audit_snap = cached.get('audit', {})

                    c_appr = cached['stats']['Appr']

                    c_deny = cached['stats']['Deny']

                    c_chg = cached['stats']['Chg']

                    c_miss = sum(1 for r in cached['rows'] if r['is_missing'])



                    logger.info(f"  ✅ Read (Cache): [{idx}/{total_revs}] {reviewer_name} (Missing:{c_miss})(A:{c_appr}, D:{c_deny}, C:{c_chg})")



                    for r in cached['rows']:

                        r_copy = r.copy()

                        st = get_row_stats(r['response'])

                        r_copy.update({

                            "Category": category, "App_Name": current_app_name,

                            "Last_Modified": remote_mod, "File_Created_Date": audit_snap.get('created_ts'),

                            "Audit_Log": audit_snap.get('log'), "File_Creator": audit_snap.get('creator'), "File_Modifier": audit_snap.get('modifier'),

                            "stats_appr": st['is_appr'], "stats_deny": st['is_deny'], "stats_chg": st['is_chg'],

                            "source_is_cache": True

                        })

                        all_responses.append(r_copy)

                    continue



                content = download_file(site_id, f"{folder_path}/{target_file['name']}")

                audit_info = get_file_audit_info(site_id, target_file["id"])



                rows, detected_col_map, map_source = read_excel_rows(
                    content, reviewer_name, target_file['name'], folder_url, app_col_map=app_col_map
                )

                if detected_col_map and not app_col_map:

                    app_col_map = detected_col_map

                    app_column_map_store[app_schema_key] = detected_col_map

                    logger.info(
                        f"  🧭 App Column Map Locked: NameIdx={detected_col_map.get('idx_user')} "
                        f"EmailIdx={detected_col_map.get('idx_email')} Source={map_source}"
                    )
                elif detected_col_map and map_source == "app-locked":

                    logger.info(
                        f"  🧭 App Column Map Reused: NameIdx={detected_col_map.get('idx_user')} "
                        f"EmailIdx={detected_col_map.get('idx_email')}"
                    )
                else:

                    logger.warning(f"  ⚠️ Column Mapping Failed: [{idx}/{total_revs}] {reviewer_name}")



                s_appr, s_deny, s_chg, miss_cnt = 0, 0, 0, 0

                clean_rows_cache = []

                final_created = audit_info.get('created_ts') or target_file.get("createdDateTime")



                for r in rows:

                    st = get_row_stats(r['response'])

                    s_appr += st['is_appr']; s_deny += st['is_deny']; s_chg += st['is_chg']

                    if r['is_missing']: miss_cnt += 1



                    clean_rows_cache.append({

                        "reviewer": r['reviewer'], "user_name": r.get('user_name', ''), "user_email": r.get('user_email', ''),

                        "response": r['response'], "details": r['details'],

                        "is_missing": r['is_missing'], "row_number": r['row_number'],

                        "file_name": r['file_name'], "folder_url": r['folder_url']

                    })



                    r.update({

                        "Category": category, "App_Name": current_app_name,

                        "Last_Modified": remote_mod, "File_Created_Date": final_created,

                        "Audit_Log": audit_info['log'], "File_Creator": audit_info['creator'], "File_Modifier": audit_info['modifier'],

                        "stats_appr": st['is_appr'], "stats_deny": st['is_deny'], "stats_chg": st['is_chg'],

                        "source_is_cache": False

                    })



                all_responses.extend(rows)



                logger.info(
                    f"  ✅ Read (Live): [{idx}/{total_revs}] {reviewer_name} "
                    f"(Rows:{len(rows)} Missing:{miss_cnt})(A:{s_appr}, D:{s_deny}, C:{s_chg})"
                )



                if rows or force_live_recheck:

                    local_cache[cache_key] = {

                        "v": CACHE_VERSION, "last_mod": remote_mod,

                        "stats": {"Appr": s_appr, "Deny": s_deny, "Chg": s_chg},

                        "audit": audit_info, "rows": clean_rows_cache

                    }

                    cache_updated = True

                    logger.info(
                        f"  💾 Cache Write: [{idx}/{total_revs}] {reviewer_name} "
                        f"(Rows:{len(clean_rows_cache)} Missing:{miss_cnt} PendingRecheck:{force_live_recheck})"
                    )



            except Exception as e:

                logger.error(f"  ❌ Error {reviewer_name}: {e}")

                errors.append({"Category": category, "App_Name": current_app_name, "reviewer": reviewer_name, "error": str(e), "folder_url": folder_url})

            finally:

                if app_ui:

                    app_ui["bar"].value = min(idx, app_ui["bar"].max)

                    app_ui["status"].value = f"{idx}/{total_revs}"



        if app_ui:
            app_ui["bar"].bar_style = "success"
            if total_revs <= 0:
                app_ui["bar"].value = 1
                app_ui["bar"].max = 1
                app_ui["status"].value = "0/0 Done"
            else:
                app_ui["bar"].value = total_revs
                app_ui["status"].value = f"{total_revs}/{total_revs} Done"

    except Exception as e:

        logger.error(f"❌ App Error: {e}")

        if app_ui:

            app_ui["bar"].bar_style = "danger"

            app_ui["status"].value = "Error"



if cache_updated:

    save_json(CACHE_FILE, local_cache)

    logger.info("💾 Cache Updated (v4.2)")



# ==========================================

# PART 4: DASHBOARD

# ==========================================

df = pd.DataFrame(all_responses)

widget_store = {}

unified_data = {}

today_str = datetime.now().strftime("%Y-%m-%d")

report_dir = f"output/{today_str}/report"

os.makedirs(report_dir, exist_ok=True)



if not df.empty:

    stats = df.groupby(["Category", "App_Name", "reviewer"]).agg(

        total_rows=("reviewer", "size"),

        missing=("is_missing", "sum"),

        approved=("stats_appr", "sum"), denied=("stats_deny", "sum"), changed=("stats_chg", "sum"),

        f_creator=("File_Creator", "first"), f_modifier=("File_Modifier", "first"), audit=("Audit_Log", "first"),

        is_cached=("source_is_cache", "all")

    ).reset_index()



    for _, row in stats.iterrows():

        key = f"{row['Category']} > {row['App_Name']}"

        if key not in unified_data:

            saved_app = manual_data_store.get(key, {})

            unified_data[key] = {

                "Category": row['Category'], "App_Name": row['App_Name'],

                "status_manual": saved_app.get("app_status", "Calculated"), "note_manual": saved_app.get("app_note", ""),

                "reviewers": {}, "stats": {"total_users": 0, "completed_users": 0}

            }



        node = unified_data[key]

        node['stats']['total_users'] += 1

        is_done = (row['missing'] == 0)

        if is_done: node['stats']['completed_users'] += 1

        status_calc = f"❌ Pending: {row['missing']}"

        if is_done: status_calc = "✅ Cached - Completed" if row['is_cached'] else "✅ Completed"



        d_style = "color:red;font-weight:bold" if row['denied'] > 0 else "color:#555"

        node['reviewers'][row['reviewer']] = {

            "status_calc": status_calc,

            "detail_html": f"Appr:{int(row['approved'])} | <span style='{d_style}'>Deny:{int(row['denied'])}</span> | Chg:{int(row['changed'])}",

            "folder_url": df[(df['App_Name'] == row['App_Name']) & (df['reviewer'] == row['reviewer'])].iloc[0].get('folder_url', '#')

        }



def build_dashboard():

    container = widgets.VBox()

    btn_export = widgets.Button(description="💾 Save Reports", button_style='success', icon='file-excel')

    btn_global = widgets.Button(description="📊 Global Report", button_style='primary')

    lbl_out = widgets.Label()



    app_widgets = []

    for app_key in sorted(unified_data.keys()):

        app_data = unified_data[app_key]

        pct = int((app_data['stats']['completed_users'] / app_data['stats']['total_users'] * 100)) if app_data['stats']['total_users'] > 0 else 0

        w_lbl = widgets.HTML(f"<b>📂 {app_key}</b> &nbsp; <span style='background:#eee; padding:2px 5px; border-radius:4px'>{pct}% Done</span>", layout=widgets.Layout(width='400px'))

        w_stat = widgets.Dropdown(options=["Calculated", "Force Completed", "Action Required"], value=app_data['status_manual'], layout=widgets.Layout(width='150px'))

        widget_store[app_key] = {"data": app_data, "w_stat": w_stat}



        rev_list = widgets.VBox([

            widgets.HBox([

                widgets.HTML(f"<a href='{rd['folder_url']}' target='_blank'>{rn}</a>", layout=widgets.Layout(width='250px')),

                widgets.HTML(rd['status_calc'], layout=widgets.Layout(width='200px')),

                widgets.HTML(rd['detail_html'])

            ]) for rn, rd in app_data['reviewers'].items()

        ], layout=widgets.Layout(margin='5px 0 5px 20px', display='none'))



        btn_tog = widgets.Button(description="➕ Show Users", layout=widgets.Layout(width='100px'))

        def create_tog(w):

            def on_tog(b):

                if w.layout.display == 'none': w.layout.display = 'block'; b.description = "➖ Hide"

                else: w.layout.display = 'none'; b.description = "➕ Show Users"

            return on_tog

        btn_tog.on_click(create_tog(rev_list))

        app_widgets.append(widgets.VBox([widgets.HBox([btn_tog, w_lbl, w_stat]), rev_list]))



    def export(b):

        b.disabled=True; b.description="Saving..."

        saved_files = []

        for app_key, widget_data in widget_store.items():

            app_name = widget_data['data']['App_Name']

            category = widget_data['data']['Category']

            app_rows = df[(df['Category'] == category) & (df['App_Name'] == app_name)].to_dict('records')



            final_data = []

            for row in app_rows:

                overwrite_st = widget_data['w_stat'].value

                if overwrite_st == "Calculated": overwrite_st = ""



                final_data.append({

                    "User Name": row.get('user_name', ''),

                    "User Email": row.get('user_email', ''),

                    "Reviewer": row['reviewer'],

                    "File Name": row.get('file_name'),

                    "Reviewer Response": row.get('response'),

                    "Details of Access Change": row.get('details'),

                    "Overwrite Status": overwrite_st,

                    "Row Num": row.get('row_number'),

                    "Audit Log": row.get('Audit_Log')

                })



            if final_data:

                safe_name = re.sub(r'[\\/*?:"<>|]', "", app_name)

                f_path = safe_excel_path(f"{report_dir}/{safe_name}_{today_str}.xlsx")

                pd.DataFrame(final_data).to_excel(f_path, index=False)
                format_export_excel(f_path)

                saved_files.append(safe_name)



        lbl_out.value = f"Saved {len(saved_files)} files."

        b.disabled=False; b.description="💾 Save Reports"



    def export_global(b):

        b.disabled = True; b.description = "Saving..."

        rows = []

        for _, r in stats.sort_values("App_Name").iterrows():

            progress = "Completed" if r['missing'] == 0 else f"{int(r['missing'])}/{int(r['total_rows'])} missing"

            rows.append({

                "Category": r['Category'],

                "App Name": r['App_Name'],

                "Reviewer": r['reviewer'],

                "Progress": progress,

                "Total Approved": int(r['approved']),

                "Total Denied": int(r['denied']),

                "Total Changed": int(r['changed'])

            })

        f_path = safe_excel_path(f"{report_dir}/Summary_Report_{today_str}.xlsx")

        pd.DataFrame(rows).to_excel(f_path, index=False)
        format_export_excel(f_path)

        lbl_out.value = f"Global report saved."

        b.disabled = False; b.description = "📊 Global Report"



    btn_export.on_click(export)

    btn_global.on_click(export_global)

    container.children = tuple([widgets.HBox([btn_export, btn_global, lbl_out])] + app_widgets)

    display(container)



if unified_data:

    build_dashboard()

else:

    print("⚠️ No data found.")


In [ ]:
# === Cell 6: Email Notification (Global App Sections by Reviewer) ===
import ipywidgets as widgets
from urllib.parse import quote
from datetime import datetime, timedelta
import requests
import pandas as pd
import re
from IPython.display import display, clear_output

# --- 1. Helper Functions ---
def get_email(name):
    if not name:
        return ""
    try:
        clean = quote(name.split("(")[0].strip())
        url = f"https://graph.microsoft.com/v1.0/users?$filter=startswith(displayName,'{clean}')&$select=mail,userPrincipalName"
        res = requests.get(url, headers=headers).json().get("value")
        if res:
            return res[0].get("mail") or res[0].get("userPrincipalName")
    except Exception:
        pass
    return ""


def fmt_date_long(iso_date_str):
    if not iso_date_str or pd.isna(iso_date_str):
        return "Unknown Date"
    try:
        dt_str = str(iso_date_str).replace("Z", "").split(".")[0]
        dt = datetime.fromisoformat(dt_str)
        return dt.strftime("%B %d, %Y")
    except Exception:
        return str(iso_date_str)


def calc_due_date_long(iso_date_str):
    if not iso_date_str or pd.isna(iso_date_str):
        return "ASAP"
    try:
        dt_str = str(iso_date_str).replace("Z", "").split(".")[0]
        dt = datetime.fromisoformat(dt_str)
        due = dt + timedelta(days=14)
        return due.strftime("%B %d, %Y")
    except Exception:
        return "ASAP"


def iso_to_date(iso_date_str):
    if not iso_date_str or pd.isna(iso_date_str):
        return None
    try:
        dt_str = str(iso_date_str).replace("Z", "").split(".")[0]
        dt = datetime.fromisoformat(dt_str)
        return dt.date()
    except Exception:
        return None


def format_date_long(date_obj):
    if not date_obj:
        return ""
    try:
        return date_obj.strftime("%B %d, %Y")
    except Exception:
        return str(date_obj)


def parse_email_list(raw_text):
    if not raw_text:
        return []
    parts = re.split(r"[,\n;]+", str(raw_text))
    return [p.strip() for p in parts if p and p.strip()]


def infer_section(file_date_raw):
    if not file_date_raw or pd.isna(file_date_raw):
        return "followup"
    try:
        dt_str = str(file_date_raw).replace("Z", "").split(".")[0]
        sent_dt = datetime.fromisoformat(dt_str)
        due_dt = sent_dt + timedelta(days=14)
        return "new" if due_dt >= datetime.now() else "followup"
    except Exception:
        return "followup"


def build_section_table(rows):
    ordered_rows = sorted(rows, key=lambda x: (x["Category"], x["App_Name"]))
    table = (
        "<table border='1' style='border-collapse:collapse; width:100%; font-size:12px; "
        "border:1px solid #d8dde6; table-layout:fixed'>"
    )
    table += (
        "<tr style='background:#eef3f8'>"
        "<th style='padding:6px; width:18%'>Quarter</th>"
        "<th style='padding:6px; width:22%'>App Name</th>"
        "<th style='padding:6px; width:18%'>Sent Date</th>"
        "<th style='padding:6px; width:18%'>Due Date</th>"
        "<th style='padding:6px; width:14%'>Users to review</th>"
        "<th style='padding:6px; width:10%'>Link</th>"
        "</tr>"
    )
    for row in ordered_rows:
        table += (
            "<tr>"
            f"<td style='padding:6px; word-wrap:break-word'>{row['Category']}</td>"
            f"<td style='padding:6px; word-wrap:break-word'>{row['App_Name']}</td>"
            f"<td style='padding:6px'>{row['sent_date']}</td>"
            f"<td style='padding:6px'>{row['due_date']}</td>"
            f"<td style='padding:6px; text-align:center'>{row['missing']}</td>"
            f"<td style='padding:6px'><a href='{row['folder_url']}'>Open</a></td>"
            "</tr>"
        )
    table += "</table>"
    return table


def build_email_html(reviewer_name, new_rows, followup_rows):
    parts = [
        f"Hi {reviewer_name},<br><br>",
        "The Information Security team is sending you an entitlement review update.<br><br>",
    ]

    if new_rows:
        parts.append("<b>1) New Applications to Review</b><br>")
        parts.append(build_section_table(new_rows))
        parts.append("<br><br>")

    if followup_rows:
        parts.append("<b>2) Follow-up on Remaining Reviews</b><br>")
        parts.append(build_section_table(followup_rows))
        parts.append("<br><br>")

    if not new_rows and not followup_rows:
        parts.append("No applications are selected for this update.<br><br>")

    parts.append(
        "Please complete these reviews as soon as possible. "
        "Your prompt assistance to this matter is appreciated.<br><br>"
        "Sincerely,<br>Apple Bank's Information Security Team"
    )
    return "".join(parts)


def resolve_subject(template, reviewer_name):
    try:
        return template.format(reviewer_name=reviewer_name)
    except Exception:
        return template


# --- 2. Data Preparation ---
if "df" not in globals() or df.empty:
    display(widgets.HTML("<h4 style='color:red'>⚠️ No data found. Run Cell 5 first.</h4>"))
else:
    pending_df = df[df["is_missing"] == True].copy()
    targets = pending_df.groupby(["Category", "App_Name", "reviewer", "folder_url"]).agg(
        missing_count=("is_missing", "count"),
        file_date_raw=("File_Created_Date", "min"),
    ).reset_index()

    notes_db = manual_data_store if "manual_data_store" in globals() else {}
    print("🔍 Looking up emails for pending reviewers...")
    email_cache = {}
    raw_data_list = []

    for _, r in targets.iterrows():
        app_key = f"{r['Category']} > {r['App_Name']}"
        reviewer_key = r["reviewer"]
        app_node = notes_db.get(app_key, {})
        if app_node.get("app_status") in ["Force Completed", "N/A"]:
            continue

        rev_override = app_node.get("reviewers", {}).get(reviewer_key, {}).get("override", "-")
        if rev_override in ["Mark Done", "Waived", "Escalated"]:
            continue

        if reviewer_key not in email_cache:
            email_cache[reviewer_key] = get_email(reviewer_key)

        raw_data_list.append(
            {
                "Category": r["Category"],
                "App_Name": r["App_Name"],
                "app_key": app_key,
                "reviewer": reviewer_key,
                "missing": int(r["missing_count"]),
                "folder_url": r["folder_url"],
                "sent_date": fmt_date_long(r["file_date_raw"]),
                "due_date": calc_due_date_long(r["file_date_raw"]),
                "file_date_raw": r["file_date_raw"],
                "email": email_cache[reviewer_key],
            }
        )

    clear_output()

    # --- 3. UI State ---
    current_year = datetime.now().year
    default_subject = f"REMINDER - {current_year} Entitlement Review Update"

    df_raw = pd.DataFrame(raw_data_list)
    row_logic_store = {}
    app_section_store = {}
    app_send_store = {}
    app_due_store = {}
    app_due_mode = {}
    app_guard = {}
    app_selector_rows = []

    # --- 4. Top Controls ---
    w_subject_tpl = widgets.Text(
        value=default_subject,
        placeholder="Supports {reviewer_name}",
        layout=widgets.Layout(width="100%"),
    )
    w_cc_global = widgets.Textarea(
        value="",
        placeholder="manager@applebank.com, team@applebank.com",
        layout=widgets.Layout(width="100%", height="70px"),
    )
    w_reply_to = widgets.Text(
        value="",
        placeholder="security-team@applebank.com",
        layout=widgets.Layout(width="100%"),
    )
    w_send_from = widgets.Text(
        value=(SENDER_EMAIL or ""),
        placeholder="shared-mailbox@applebank.com",
        layout=widgets.Layout(width="100%"),
    )
    btn_refresh = widgets.Button(
        description="🔄 Refresh List",
        button_style="info",
        layout=widgets.Layout(width="180px"),
    )
    btn_send_all = widgets.Button(
        description="🔥 Send All (Batch)",
        button_style="danger",
        layout=widgets.Layout(width="200px"),
    )

    app_selector_box = widgets.VBox()
    reviewer_box = widgets.VBox()

    def derive_dates_for_row(row):
        app_key = row.get("app_key")
        send_widget = app_send_store.get(app_key)
        due_widget = app_due_store.get(app_key)

        send_override = send_widget.value if send_widget else None
        due_override = due_widget.value if due_widget else None

        sent_date = send_override or iso_to_date(row.get("file_date_raw"))
        sent_str = format_date_long(sent_date) if sent_date else "Unknown Date"

        due_date = due_override
        if not due_date and sent_date:
            due_date = sent_date + timedelta(days=14)
        due_str = format_date_long(due_date) if due_date else "ASAP"
        return sent_str, due_str


    def with_derived_dates(rows):
        out = []
        for r in rows:
            sent_str, due_str = derive_dates_for_row(r)
            rr = dict(r)
            rr["sent_date"] = sent_str
            rr["due_date"] = due_str
            out.append(rr)
        return out


    def build_app_selector_state():
        nonlocal_df = df_raw.copy()
        if nonlocal_df.empty:
            app_selector_box.children = (widgets.HTML("<i>No apps available.</i>"),)
            return

        app_summary = nonlocal_df.groupby(["app_key", "Category", "App_Name"]).agg(
            reviewer_count=("reviewer", "nunique"),
            missing_total=("missing", "sum"),
            file_date_raw=("file_date_raw", "min"),
        ).reset_index().sort_values(["Category", "App_Name"])

        current_section_values = {k: v.value for k, v in app_section_store.items()}
        current_send_values = {k: v.value for k, v in app_send_store.items()}
        current_due_values = {k: v.value for k, v in app_due_store.items()}
        current_due_mode = dict(app_due_mode)

        app_section_store.clear()
        app_send_store.clear()
        app_due_store.clear()
        app_due_mode.clear()
        app_guard.clear()

        rows = []

        for _, row in app_summary.iterrows():
            app_key = row["app_key"]
            default_value = current_section_values.get(app_key, infer_section(row["file_date_raw"]))
            w_section = widgets.Dropdown(
                options=[("New", "new"), ("Follow-up", "followup"), ("Skip", "skip")],
                value=default_value,
                layout=widgets.Layout(width="140px"),
            )
            app_section_store[app_key] = w_section

            send_default = current_send_values.get(app_key)
            if send_default is None:
                send_default = iso_to_date(row["file_date_raw"])

            due_default = current_due_values.get(app_key)
            if due_default is None and send_default:
                due_default = send_default + timedelta(days=14)

            w_send = widgets.DatePicker(value=send_default, layout=widgets.Layout(width="160px"))
            w_send.description = "Send"
            w_send.style.description_width = "45px"

            w_due = widgets.DatePicker(value=due_default, layout=widgets.Layout(width="160px"))
            w_due.description = "Due"
            w_due.style.description_width = "45px"

            app_send_store[app_key] = w_send
            app_due_store[app_key] = w_due
            app_due_mode[app_key] = current_due_mode.get(app_key, "auto")
            app_guard[app_key] = False

            info_html = (
                f"<b>Quarter: {row['Category']} &gt; {row['App_Name']}</b><br>"
                f"<span style='color:#555'>Reviewers: {int(row['reviewer_count'])}</span> &nbsp;"
                f"<span style='color:#555'>Pending Items: {int(row['missing_total'])}</span>"
            )
            rows.append(
                widgets.VBox(
                    [
                        widgets.HBox([w_section, w_send, w_due], layout=widgets.Layout(align_items="center")),
                        widgets.HTML(info_html, layout=widgets.Layout(width="100%")),
                    ],
                    layout=widgets.Layout(border="1px solid #ececec", padding="6px", margin="2px 0"),
                )
            )

        app_selector_rows[:] = rows
        app_selector_box.children = (
            widgets.HTML(
                "<div style='font-size:12px;color:#555'>Set each app once here. The selection applies to <b>all reviewers</b>.</div>"
            ),
            widgets.VBox(rows, layout=widgets.Layout(width="100%")),
        )

    def get_sectioned_rows(reviewer_rows):
        new_rows, followup_rows = [], []
        for row in reviewer_rows:
            selected_section = app_section_store.get(row["app_key"])
            section_value = selected_section.value if selected_section else "followup"
            if section_value == "new":
                new_rows.append(row)
            elif section_value == "followup":
                followup_rows.append(row)
        return with_derived_dates(new_rows), with_derived_dates(followup_rows)

    def update_preview(key):
        state = row_logic_store[key]
        new_rows, followup_rows = get_sectioned_rows(state["rows"])
        new_missing = sum(r["missing"] for r in new_rows)
        followup_missing = sum(r["missing"] for r in followup_rows)
        state["w_summary"].value = (
            f"<span style='color:#0b6'><b>New:</b> {len(new_rows)} app(s), {new_missing} item(s)</span> &nbsp; "
            f"<span style='color:#9a6700'><b>Follow-up:</b> {len(followup_rows)} app(s), {followup_missing} item(s)</span>"
        )
        body_html = build_email_html(state["reviewer"], new_rows, followup_rows)
        state["body_html"] = body_html
        state["w_preview"].value = (
            "<div style='font-family:sans-serif; font-size:12px; line-height:1.45; padding:8px; color:#333'>"
            f"{body_html}"
            "</div>"
        )

    def refresh_all_previews(change=None):
        for key in list(row_logic_store.keys()):
            update_preview(key)

    def build_reviewer_cards():
        row_logic_store.clear()

        if df_raw.empty:
            reviewer_box.children = (widgets.HTML("<h3>✅ No pending emails to send!</h3>"),)
            return

        grouped = df_raw.sort_values(["reviewer", "Category", "App_Name"]).groupby(["reviewer", "email"])
        cards = []

        for idx, ((reviewer, email), group) in enumerate(grouped):
            key = f"{reviewer}_{idx}"
            reviewer_rows = group.to_dict("records")
            email_color = "#0078d4" if email else "red"

            w_chk = widgets.Checkbox(value=True, layout=widgets.Layout(width="30px"))
            w_email = widgets.Text(value=email or "", placeholder="Email", layout=widgets.Layout(width="100%"))
            w_btn = widgets.Button(description="🚀 Send", button_style="warning", layout=widgets.Layout(width="90px"))
            w_summary = widgets.HTML(layout=widgets.Layout(width="100%"))
            w_preview = widgets.HTML(
                value="",
                layout=widgets.Layout(width="100%", border="1px solid #d8dde6"),
            )
            w_detail = widgets.HTML(
                value=(
                    f"<b>👤 {reviewer}</b><br>"
                    f"<span style='color:{email_color}'>{email or '(No Email)'}</span><br>"
                    f"<span style='color:#666'>Fetched Apps: {len(reviewer_rows)}</span>"
                ),
                layout=widgets.Layout(width="260px"),
            )

            row_logic_store[key] = {
                "reviewer": reviewer,
                "rows": reviewer_rows,
                "w_chk": w_chk,
                "w_email": w_email,
                "w_btn": w_btn,
                "w_summary": w_summary,
                "w_preview": w_preview,
                "body_html": "",
            }

            def make_sender(k):
                def on_send(_):
                    state = row_logic_store[k]
                    target_email = state["w_email"].value.strip()
                    if not target_email:
                        state["w_btn"].description = "No Email"
                        return

                    new_rows, followup_rows = get_sectioned_rows(state["rows"])
                    if not new_rows and not followup_rows:
                        state["w_btn"].button_style = "danger"
                        state["w_btn"].description = "No Apps"
                        return

                    state["w_btn"].disabled = True
                    state["w_btn"].description = "..."
                    try:
                        update_preview(k)
                        final_subject = resolve_subject(w_subject_tpl.value, state["reviewer"])
                        sender_mailbox = (w_send_from.value or "").strip() or (SENDER_EMAIL or "").strip()
                        if not sender_mailbox:
                            state["w_btn"].button_style = "danger"
                            state["w_btn"].description = "No Sender"
                            return
                        url = f"https://graph.microsoft.com/v1.0/users/{quote(sender_mailbox)}/sendMail"
                        to_list = [{"emailAddress": {"address": target_email}}]
                        cc_list = [{"emailAddress": {"address": e}} for e in parse_email_list(w_cc_global.value)]
                        reply_to_list = [{"emailAddress": {"address": e}} for e in parse_email_list(w_reply_to.value)]

                        message_payload = {
                            "subject": final_subject,
                            "body": {"contentType": "HTML", "content": state["body_html"]},
                            "toRecipients": to_list,
                        }
                        if cc_list:
                            message_payload["ccRecipients"] = cc_list
                        if reply_to_list:
                            message_payload["replyTo"] = reply_to_list

                        payload = {"message": message_payload}
                        response = requests.post(
                            url,
                            headers={**headers, "Content-Type": "application/json"},
                            json=payload,
                        )
                        if response.status_code == 202:
                            state["w_btn"].button_style = "success"
                            state["w_btn"].description = "Sent"
                            state["w_chk"].value = False
                        else:
                            state["w_btn"].button_style = "danger"
                            state["w_btn"].description = "Fail"
                            print(response.text)
                    except Exception as e:
                        print(e)
                        state["w_btn"].button_style = "danger"
                    finally:
                        if state["w_btn"].description != "Sent":
                            state["w_btn"].disabled = False
                return on_send

            sender_func = make_sender(key)
            w_btn.on_click(sender_func)
            row_logic_store[key]["trigger_send"] = sender_func

            header_row = widgets.HBox(
                [
                    w_chk,
                    w_detail,
                    widgets.VBox(
                        [
                            widgets.HTML("<b>Summary</b>"),
                            w_summary,
                        ],
                        layout=widgets.Layout(flex="1", padding="0 12px"),
                    ),
                    widgets.VBox(
                        [
                            widgets.HTML("<b>Send To</b>"),
                            w_email,
                            w_btn,
                        ],
                        layout=widgets.Layout(width="230px"),
                    ),
                ],
                layout=widgets.Layout(align_items="flex-start"),
            )

            cards.append(
                widgets.VBox(
                    [
                        header_row,
                        widgets.HTML("<div style='margin:8px 0 4px 0'><b>Email Preview</b></div>"),
                        w_preview,
                    ],
                    layout=widgets.Layout(border="1px solid #cfd8e3", margin="8px 0", padding="10px"),
                )
            )

            update_preview(key)

        reviewer_box.children = tuple(cards)

    def set_all_app_sections(value):
        for dropdown in app_section_store.values():
            dropdown.value = value
        refresh_all_previews()

    def trigger_batch_send(_):
        btn_send_all.disabled = True
        btn_send_all.description = "Sending..."
        count = 0
        for key, state in row_logic_store.items():
            if state["w_chk"].value and state["w_btn"].description != "Sent":
                state["trigger_send"](state["w_btn"])
                count += 1
        btn_send_all.disabled = False
        btn_send_all.description = f"🔥 Send All (Batch) - Done {count}"

    def render_all(_=None):
        build_app_selector_state()
        for dropdown in app_section_store.values():
            dropdown.observe(refresh_all_previews, names="value")

        for app_key, send_picker in app_send_store.items():
            def _on_send(change, app_key=app_key):
                if change.get("name") != "value":
                    return
                if app_due_mode.get(app_key, "auto") == "auto":
                    send_val = change.get("new")
                    if send_val:
                        due_picker = app_due_store.get(app_key)
                        if due_picker:
                            app_guard[app_key] = True
                            try:
                                due_picker.value = send_val + timedelta(days=14)
                            finally:
                                app_guard[app_key] = False
                refresh_all_previews()

            send_picker.observe(_on_send, names="value")

        for app_key, due_picker in app_due_store.items():
            def _on_due(change, app_key=app_key):
                if change.get("name") != "value":
                    return
                if app_guard.get(app_key):
                    return
                app_due_mode[app_key] = "manual"
                refresh_all_previews()

            due_picker.observe(_on_due, names="value")
        build_reviewer_cards()
        refresh_all_previews()

    btn_all_new = widgets.Button(description="All New", layout=widgets.Layout(width="90px"))
    btn_all_followup = widgets.Button(description="All Follow-up", layout=widgets.Layout(width="120px"))
    btn_all_skip = widgets.Button(description="All Skip", layout=widgets.Layout(width="90px"))
    btn_all_new.on_click(lambda _: set_all_app_sections("new"))
    btn_all_followup.on_click(lambda _: set_all_app_sections("followup"))
    btn_all_skip.on_click(lambda _: set_all_app_sections("skip"))

    btn_refresh.on_click(render_all)
    btn_send_all.on_click(trigger_batch_send)

    render_all()

    top_panel = widgets.VBox(
        [
            widgets.HTML("<h3>📧 Email Notification Center</h3>"),
            widgets.HTML(
                "<div style='font-size:12px;color:#555'>Classify each fetched app once at the top. Those labels apply to every reviewer email.</div>"
            ),
            widgets.HBox([btn_refresh, btn_send_all]),
            widgets.HTML("<hr style='margin:8px 0'>"),
            widgets.HTML("<b>Send From</b>"),
            w_send_from,
            widgets.HTML("<div style='font-size:11px;color:#777'>Tip: enter a shared mailbox UPN/email you have <i>Send As</i> or <i>Send on behalf</i> rights for.</div>"),
            widgets.HTML("<b>Subject</b>"),
            w_subject_tpl,
            widgets.HTML("<div style='font-size:11px;color:#777'>Optional placeholder: <code>{reviewer_name}</code></div>"),
            widgets.HTML("<b>Global CC</b>"),
            w_cc_global,
            widgets.HTML("<b>Reply-To</b>"),
            w_reply_to,
        ],
        layout=widgets.Layout(background_color="#f0f4f8", padding="10px", border="1px solid #ccc", margin="0 0 10px 0"),
    )

    app_panel = widgets.VBox(
        [
            widgets.HTML("<h4 style='margin:0'>App Sections (Global)</h4>"),
            widgets.HBox([btn_all_new, btn_all_followup, btn_all_skip]),
            app_selector_box,
        ],
        layout=widgets.Layout(background_color="#fbfcfe", padding="10px", border="1px solid #d8dde6", margin="0 0 12px 0"),
    )

    display(top_panel, app_panel, reviewer_box)
